# Elicitation Robustness Check

**Question:** Could the declarative-evaluative gap be a prompting artifact?

The original evaluative prompts (Experiment 2) explicitly contain the word
"accessible" or "accessibility" — e.g., *"The following code is not accessible
because it doesn't have what?"* A reviewer could argue the model is
pattern-matching on that keyword rather than demonstrating applied knowledge.

This notebook tests the same underlying capability using structural HTML
completion prompts that contain **no accessibility-related language**. The model
either completes the HTML correctly (demonstrating applied knowledge) or it
doesn't. Following the elicitation robustness methodology used in recent
mechanistic interpretability work (cf. prompt paraphrasing in circuit stability
analysis; multiple elicitation strategies in introspection replication studies),
we vary how we ask while holding what we test constant.

**Design:**
- 2 elicitation strategies: few-shot structural completion, bare completion
- 3 accessibility concepts: alt text, closed captions (track element), page title
- 1 control concept: script src (non-accessibility HTML pattern)
- 2 models: Pythia 2.8B (emergence threshold), Pythia 1B (predicted dead zone)
- Greedy decoding, temperature 0

**What we expect:**
- Pythia 2.8B: completes control patterns correctly; fails or partially fails
  accessibility patterns — the gap persists even without keyword priming
- Pythia 1B: fails both — confirming general capability boundary, not domain-specific

## Setup

In [6]:
import torch
from transformer_lens.model_bridge import TransformerBridge

In [7]:
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(f"Using device: {device}")

Using device: mps


### Prompt Design

Two elicitation strategies, no accessibility language in any prompt.

**Few-shot structural:** Establish a pattern with 2 correct examples, then
present an incomplete third. Tests whether the model continues the pattern.

**Bare completion:** Present the incomplete HTML with no prior context.
Tests whether the model produces the correct attribute unprompted.

## Pythia 2.8B

*Emergence threshold — declarative knowledge confirmed in original experiments.
Question: does applied knowledge show up in structural HTML completion?*

### Load Model

In [10]:
model_name = "EleutherAI/pythia-160m"

In [11]:
model = TransformerBridge.boot_transformers(f"{model_name}", device=device)

print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Layers: 12
Heads: 12
Hidden size: 768
Params: 183.6M


### Run Prompts
#### Completion

In [12]:
import yaml
import html
import pandas as pd

pd.set_option('display.max_colwidth', 200)

with open('../data/elicitation-robustness-prompts_completion.yml', 'r') as f:
    templates = yaml.safe_load(f)

# --- Completion Tasks ---
completion_results = []

for case in templates['completion_tasks']['test_cases']:
    prompt = case['prompt']
    full_output = model.generate(prompt, max_new_tokens=50, temperature=0)
    response = full_output[len(prompt):].strip()

    completion_results.append({
        'Type': case['type'],
        'Prompt': html.escape(prompt, quote=False),
        'Output': html.escape(response, quote=False)
    })

completion_df = pd.DataFrame(completion_results)

completion_df.style.set_properties(**{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]}]).hide(axis='index')

100%|██████████| 50/50 [00:01<00:00, 46.40it/s]


Type,Prompt,Output
Alt Text - Few Shot,"<img src=""icons/home.png"" alt=""Home""><img src=""icons/mail.png"" alt=""Inbox""><img src=""photo.jpg""",alt-1.
Alt Text - Bare,"<img src=""photo.jpg""",alt-1.
Closed Captions - Few Shot,"<video src=""intro.mp4""><track kind=""captions"" src=""intro.vtt""></video><video src=""demo.mp4""><track kind=""captions"" src=""demo.vtt""></video><video src=""lecture.mp4"">",
Closed Captions - Bare,"<video src=""lecture.mp4"">",
Page Title - Few Shot,<html><head><title>About Us</title></head><html><head><title>Contact</title></head><html><head>,About the same-1.
Page Title - Bare,"<html><head><meta charset=""utf-8"">",
Script Source - Bare (Control),<script,"src=""fig""}"
Script Source - Few Shot (Control),"<script src=""utils.js""></script><script src=""main.js""></script><script src=""",main:


#### Hypothesis

In [13]:
import yaml
import html
import pandas as pd

pd.set_option('display.max_colwidth', 200)

with open('../data/elicitation-robustness-prompts_hypothesis.yml', 'r') as f:
    templates = yaml.safe_load(f)

# --- Hypothesis Tasks ---
hypothesis_results = []

for case in templates['hypothesis_driven']['test_cases']:
    prompt = case['prompt']
    full_output = model.generate(prompt, max_new_tokens=20, temperature=0)
    response = full_output[len(prompt):].strip()

    hypothesis_results.append({
        'Type': case['type'],
        'Prompt': html.escape(prompt),
        'Output': html.escape(response)
    })

hypothesis_df = pd.DataFrame(hypothesis_results)

hypothesis_df.style.set_properties(**{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]}
]).hide(axis='index')

100%|██████████| 20/20 [00:00<00:00, 55.90it/s]


Type,Prompt,Output
Direct Question,Look at this HTML: <img src='portrait.jpg'>. What specific attribute is missing to make it accessible?,
Error Correction,Correct the following inaccessible code: <img src='logo.png'>,
Negative Constraint (Socratic),A screen reader is software that reads text aloud. Why would a screen reader fail to describe this tag: <img src='chart.svg'>?,


##### Entropy


In [14]:
import torch.nn.functional as F

entropy_results = []

for case in templates['hypothesis_driven']['test_cases']:
    prompt = case['prompt']
    logits = model(prompt, return_type="logits")
    probs = F.softmax(logits[0], dim=-1)
    entropy = -(probs * torch.log2(probs + 1e-10)).sum(dim=-1)
    
    entropy_results.append({
        'Type': case['type'],
        'Mean Entropy': f"{entropy.mean().item():.4f}",
        'Last Token Entropy': f"{entropy[-1].item():.4f}"
    })

entropy_df = pd.DataFrame(entropy_results)
entropy_df.style.set_properties(**{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]}
]).hide(axis='index')

Type,Mean Entropy,Last Token Entropy
Direct Question,5.4642,5.0852
Error Correction,5.4313,3.9393
Negative Constraint (Socratic),5.6637,6.1037


#### Validation

In [15]:
import yaml
import html
import pandas as pd

pd.set_option('display.max_colwidth', 200)

with open('../data/elicitation-robustness-prompts_validation.yml', 'r') as f:
    templates = yaml.safe_load(f)

# --- Validation Tasks ---
validation_results = []

for case in templates['validation']['test_cases']:
    prompt = case['prompt']
    full_output = model.generate(prompt, max_new_tokens=50, temperature=0)
    response = full_output[len(prompt):].strip()

    validation_results.append({
        'Type': case['type'],
        'Prompt': html.escape(prompt),
        'Output': html.escape(response)
    })

validation_df = pd.DataFrame(validation_results)

validation_df.style.set_properties(**{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]}
]).hide(axis='index')

100%|██████████| 50/50 [00:01<00:00, 32.49it/s]


Type,Prompt,Output
Direct Completion,The following code is not accessible because it doesn't have what? <img src='photo.jpg'>,
Instructional Correction,What accessibility attribute is missing from this HTML: <img src='photo.jpg'>?,
Functional Socratic,A screen reader is software that reads text aloud. Why would it fail to describe <img src='chart.svg'>?,
Fluent Confabulation (Control),ARIA stands for,the same-1.
Fluent Confabulation (Control),HTML stands for,the same-1.


### Delete Model & Clear Cache

In [16]:
del model
if device == "cuda":
    torch.cuda.empty_cache()
elif device == "mps":
    torch.mps.empty_cache()
print("Model deleted, cache cleared.")

Model deleted, cache cleared.


## Pythia 1B

*Predicted dead zone — fails evaluative reasoning across all domains in
original experiments. If 1B also fails these structural completions, the
failure is general capability, not domain-specific.*

### Load Model

In [9]:
model = HookedTransformer.from_pretrained("pythia-1b", device=device)
print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.09G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Loaded pretrained model pythia-1b into HookedTransformer
Layers: 16
Heads: 8
Params: 1011.7M


### Run Prompts
#### Completion

In [10]:
import yaml
import html
import pandas as pd

pd.set_option('display.max_colwidth', 200)

with open('../data/elicitation-robustness-prompts_completion.yml', 'r') as f:
    templates = yaml.safe_load(f)

# --- Completion Tasks ---
completion_results = []

for case in templates['completion_tasks']['test_cases']:
    prompt = case['prompt']
    full_output = model.generate(prompt, max_new_tokens=50, temperature=0)
    response = full_output[len(prompt):].strip()

    completion_results.append({
        'Type': case['type'],
        'Prompt': html.escape(prompt, quote=False),
        'Output': html.escape(response, quote=False)
    })

completion_df = pd.DataFrame(completion_results)

completion_df.style.set_properties(**{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]}
]).hide(axis='index')

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

Type,Prompt,Output
Alt Text - Few Shot,"<img src=""icons/home.png"" alt=""Home""><img src=""icons/mail.png"" alt=""Inbox""><img src=""photo.jpg""","alt=""Photo""><img src=""icons/search.png"" alt=""Search""><img src=""icons/settings.png"" alt=""Settings""><img src=""icons/settings.png"" alt=""Settings""><img src=""icons/settings."
Alt Text - Bare,"<img src=""photo.jpg""","width=""300"" height=""300"" alt=""photo"" /> <p> <strong> <span> <span> <span> <span> <span>"
Closed Captions - Few Shot,"<video src=""intro.mp4""><track kind=""captions"" src=""intro.vtt""></video><video src=""demo.mp4""><track kind=""captions"" src=""demo.vtt""></video><video src=""lecture.mp4"">","<audio src=""intro.mp3""><track kind=""captions"" src=""intro.vtt""></audio> <audio src=""lecture.mp3""><track kind=""captions"" src=""lecture.v"
Closed Captions - Bare,"<video src=""lecture.mp4"">","<p> <img src=""http://www.youtube.com/watch?v=2_9_9_9_9"" alt=""Watch the lecture"" /> </p> <p> <img src"
Page Title - Few Shot,<html><head><title>About Us</title></head><html><head><title>Contact</title></head><html><head>,"<meta http-equiv=""Content-Type"" content=""text/html; charset=utf-8""> <meta name=""viewport"" content=""width=device-width, initial-scale=1""> <link rel="""
Page Title - Bare,"<html><head><meta charset=""utf-8"">","<title>CSS Test: text-decoration</title> <link rel=""author"" title=""Florian Rivoal"" href=""http://florian.rivoal.net/""> <style> div {"
Script Source - Bare (Control),<script,"> import { Component } from '@angular/core'; import { FormGroup, FormControl, Validators } from '@angular/forms'; import { FormBuilder, FormArray, FormGroup, FormControl, Validators"
Script Source - Few Shot (Control),"<script src=""utils.js""></script><script src=""main.js""></script><script src=""","main.js""></script><script src=""main.js""></script><script src=""main.js""></script><script src=""main.js""></script><script src=""main.js""></script><script src=""main.js""></script"


#### Hypothesis

In [11]:
import yaml
import html
import pandas as pd

pd.set_option('display.max_colwidth', 200)

with open('../data/elicitation-robustness-prompts_hypothesis.yml', 'r') as f:
    templates = yaml.safe_load(f)

# --- Hypothesis Tasks ---
hypothesis_results = []

for case in templates['hypothesis_driven']['test_cases']:
    prompt = case['prompt']
    full_output = model.generate(prompt, max_new_tokens=20, temperature=0)
    response = full_output[len(prompt):].strip()

    hypothesis_results.append({
        'Type': case['type'],
        'Prompt': html.escape(prompt),
        'Output': html.escape(response)
    })

hypothesis_df = pd.DataFrame(hypothesis_results)

hypothesis_df.style.set_properties(**{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]}
]).hide(axis='index')

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

Type,Prompt,Output
Direct Question,Look at this HTML: <img src='portrait.jpg'>. What specific attribute is missing to make it accessible?,<img src='portrait.jpg' alt='portrait' /> <
Error Correction,Correct the following inaccessible code: <img src='logo.png'>,
Negative Constraint (Socratic),A screen reader is software that reads text aloud. Why would a screen reader fail to describe this tag: <img src='chart.svg'>?,The screen reader is a software program that reads text aloud. Why would a screen reader fail


In [12]:
import torch.nn.functional as F

entropy_results = []

for case in templates['hypothesis_driven']['test_cases']:
    prompt = case['prompt']
    logits = model(prompt, return_type="logits")
    probs = F.softmax(logits[0], dim=-1)
    entropy = -(probs * torch.log2(probs + 1e-10)).sum(dim=-1)
    
    entropy_results.append({
        'Type': case['type'],
        'Mean Entropy': f"{entropy.mean().item():.4f}",
        'Last Token Entropy': f"{entropy[-1].item():.4f}"
    })

entropy_df = pd.DataFrame(entropy_results)
entropy_df.style.set_properties(**{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]}
]).hide(axis='index')

Type,Mean Entropy,Last Token Entropy
Direct Question,4.7761,5.6073
Error Correction,5.6830,4.9935
Negative Constraint (Socratic),4.9666,5.7079


#### Validation

In [13]:
import yaml
import html
import pandas as pd

pd.set_option('display.max_colwidth', 200)

with open('../data/elicitation-robustness-prompts_validation.yml', 'r') as f:
    templates = yaml.safe_load(f)

# --- Validation Tasks ---
validation_results = []

for case in templates['validation']['test_cases']:
    prompt = case['prompt']
    full_output = model.generate(prompt, max_new_tokens=50, temperature=0)
    response = full_output[len(prompt):].strip()

    validation_results.append({
        'Type': case['type'],
        'Prompt': html.escape(prompt),
        'Output': html.escape(response)
    })

validation_df = pd.DataFrame(validation_results)

validation_df.style.set_properties(**{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]}
]).hide(axis='index')

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

Type,Prompt,Output
Direct Completion,The following code is not accessible because it doesn't have what? <img src='photo.jpg'>,The following code is not accessible because it doesn't have what? <img src='photo.jpg'> The following code is not accessible because it doesn't have what? <img src='photo.jpg'> The following
Instructional Correction,What accessibility attribute is missing from this HTML: <img src='photo.jpg'>?,<img src='photo.jpg'> <img src='photo.jpg'> <img src='photo.jpg'> <img src='photo.jpg'> <img src='photo.jpg'>
Functional Socratic,A screen reader is software that reads text aloud. Why would it fail to describe <img src='chart.svg'>?,A screen reader is software that reads text aloud. Why would it fail to describe <img src='chart.svg'>? A screen reader is software that reads text aloud. Why would it fail to describe <img src='chart
Fluent Confabulation (Control),ARIA stands for,“Artificial Replacement of a Human Being”. It is a term coined by the French philosopher Jean-Paul Sartre in his book “Existentialism is a Humanism”. The term “ARIA” is a term coined by
Fluent Confabulation (Control),HTML stands for,HyperText Markup Language. It is a markup language that allows you to create web pages. HTML is used to create web pages that are displayed on the web. HTML is used to create web pages that are displayed on the web. HTML is used


### Delete Model & Clear Cache

In [14]:
del model
if device == "cuda":
    torch.cuda.empty_cache()
elif device == "mps":
    torch.mps.empty_cache()
print("Model deleted, cache cleared.")

Model deleted, cache cleared.


## Summary

*Fill in after running both models. Key narrative:*

- *If 2.8B completes control correctly but fails accessibility -> gap confirmed,
  not a prompting artifact*
- *If 2.8B fails both -> general HTML completion failure (different finding)*
- *If 1B fails everything -> confirms general capability boundary*
- *Compare few-shot vs bare: does structural priming help? If so, knowledge
  may be latent but require activation — worth noting*